# Remove Null values and Deduplication

## 1.Merge Deduplicated Customer Records into Silver Layer

In [0]:
%python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

customers_silver_window = Window.partitionBy("id").orderBy(col("created_ts").desc())

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.orderBy("date")
sales_data = sales_data.withColumn("row_number", row_number().over(window))
sales_data.show()


In [0]:
%sql
INSERT INTO silver.customers
SELECT *
FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY created_ts DESC) AS rn
  FROM customers_
  WHERE cst_id IS NOT NULL AND cst_key IS NOT NULL
) AS dedup
WHERE rn = 1;
 



## 2.Merge Deduplicated Products Records into Silver Layer

In [0]:
%sql
MERGE INTO silver.products AS target
USING (
  SELECT *
  FROM (
    SELECT *, 
           ROW_NUMBER() OVER (PARTITION BY prd_id ORDER BY  prd_start_dt DESC) AS rn
    FROM bronze.products
  ) AS m
  WHERE rn = 1
) AS source
ON target.prd_id = source.prd_id
WHEN MATCHED THEN
  UPDATE SET
    target.prd_key = source.prd_key,
    target.prd_nm = source.prd_nm,
    target.prd_cost = source.prd_cost,
    target.prd_line = source.prd_line,
    target.prd_start_dt = source.prd_start_dt,
    target.prd_end_dt = source.prd_end_dt,
   target.created_ts = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (
    prd_id, prd_key, prd_nm, prd_cost, prd_line, prd_start_dt, prd_end_dt, created_at
  )
  VALUES (
    source.prd_id, source.prd_key, source.prd_nm, source.prd_cost, source.prd_line,
    source.prd_start_dt, source.prd_end_dt, source.created_at
  );




## 3.Merge Deduplicated sales Records into Silver Layer

In [0]:
%sql
MERGE INTO silver.sales AS target
USING (
  SELECT *
  FROM (
    SELECT
      TRIM(sls_ord_num)   AS sls_ord_num,
      TRIM(sls_prod_key)  AS sls_prod_key,
      sls_cust_id,
      sls_order_dt,
      sls_ship_dt,
      sls_due_dt,
      sls_sales,
      sls_quantity,
      sls_price,
      created_at,
      current_timestamp() AS processed_at,
      ROW_NUMBER() OVER (PARTITION BY sls_ord_num ORDER BY created_at DESC) AS rn
    FROM bronze.sales
    WHERE 
      sls_ord_num IS NOT NULL AND
      sls_prod_key IS NOT NULL AND
      sls_cust_id IS NOT NULL AND
      sls_order_dt IS NOT NULL AND
      sls_ship_dt IS NOT NULL AND
      sls_due_dt IS NOT NULL AND
      sls_quantity IS NOT NULL
  ) AS deduped
  WHERE rn = 1
) AS source
ON target.sls_ord_num = source.sls_ord_num  
WHEN MATCHED THEN 
UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
WHEN NOT MATCHED BY SOURCE THEN 
  DELETE ;

## 4.Merge Deduplicated Customer_erp Records into Silver Layer

In [0]:
%sql
MERGE INTO silver.customers_erp AS target
USING (
  SELECT *
  FROM (
    SELECT
      TRIM(cid) AS cid,
      bdate,
      TRIM(gen) AS gen,
      created_at,
      current_timestamp() AS processed_at,
      ROW_NUMBER() OVER (PARTITION BY cid ORDER BY created_at DESC) AS rn
    FROM bronze.customers_erp
    WHERE cid IS NOT NULL
  ) AS ranked
  WHERE rn = 1
) AS source
ON target.cid = source.cid
WHEN MATCHED THEN 
UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
WHEN NOT MATCHED BY SOURCE THEN 
  DELETE ;

## 5.Merge Deduplicated Location Records into Silver Layer

In [0]:
%sql
MERGE INTO silver.location_erp AS target
USING (
  SELECT *
  FROM (
    SELECT
      TRIM(cid) AS cid,
      TRIM(cntry) AS cntry,
      created_at,
      current_timestamp() AS processed_at,
      ROW_NUMBER() OVER (PARTITION BY cid ORDER BY created_at DESC) AS rn
    FROM bronze.location_erp
  ) AS ranked
  WHERE rn = 1
) AS source
ON target.cid = source.cid
WHEN MATCHED THEN 
UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
WHEN NOT MATCHED BY SOURCE THEN 
  DELETE ;




## 6.Merge Deduplicated Category Records into Silver Layer

In [0]:
%sql
MERGE INTO silver.category AS target
USING (
  SELECT *
  FROM (
    SELECT
      TRIM(id) AS id,
      TRIM(cat) AS cat,
      TRIM(subcat) AS subcat,
      TRIM(maintenance) AS maintenance,
      created_at,
      current_timestamp() AS processed_at,
      ROW_NUMBER() OVER (PARTITION BY id ORDER BY created_at DESC) AS rn
    FROM bronze.category
    WHERE 
      id IS NOT NULL AND
      cat IS NOT NULL AND
      subcat IS NOT NULL AND
      maintenance IS NOT NULL
  ) AS ranked
  WHERE rn = 1
) AS source
ON target.id = source.id
WHEN MATCHED THEN 
UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
WHEN NOT MATCHED BY SOURCE THEN 
  DELETE ;